# 01 — Build Daily Feature Dataset

Produces `data/daily_features.csv` — one row per (ticker, trading day) with:
- Market features: open, high, low, close, volume, return_1d, return_3d, rolling_volatility_5d
- Sentiment features: article_count, sentiment_mean, sentiment_max, sentiment_min, sentiment_std, positive_count, negative_count

**No future leakage** — sentiment is aggregated only from articles published on or before each trading day.

In [ ]:
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm

# replacing vader with finBERT llm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FILES_DIR = PROJECT_ROOT / 'files'
DATA_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dataset_builder import DatasetPaths, build_dataset

print('Project root:', PROJECT_ROOT)


## 1. Load Market Data

In [ ]:
TICKERS = ['NVDA', 'GOOG', 'TSLA']
TICKER_FILE_MAP = {
    'NVDA': FILES_DIR / 'NVDA_q4_2025.csv',
    'GOOG': FILES_DIR / 'GOOG_q4_2025.csv',
    'TSLA': FILES_DIR / 'TSLA_q4_2025.csv',
}

market_frames = []
for ticker, path in TICKER_FILE_MAP.items():
    df = pd.read_csv(path, parse_dates=['Date'])
    df = df.rename(columns={'Date': 'date'})
    df['ticker'] = ticker
    # Keep only raw OHLCV — we'll recompute derived features cleanly
    df = df[['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
    market_frames.append(df)

market = pd.concat(market_frames, ignore_index=True)
market = market.sort_values(['ticker', 'date']).reset_index(drop=True)
market['date'] = pd.to_datetime(market['date'])

print('Market data shape:', market.shape)
print('Date range:', market['date'].min().date(), '→', market['date'].max().date())
market.head()

## 2. Compute Market Features

In [ ]:
def compute_market_features(df):
    df = df.copy().sort_values('date')
    df['return_1d'] = df['close'].pct_change(1)
    df['return_3d'] = df['close'].pct_change(3)
    df['rolling_volatility_5d'] = df['return_1d'].rolling(5).std()
    return df

market = market.groupby('ticker', group_keys=False).apply(compute_market_features)
market = market.reset_index(drop=True)

print('Sample NVDA:')
market[market['ticker'] == 'NVDA'][['date', 'close', 'return_1d', 'return_3d', 'rolling_volatility_5d']].head(8)

## 3. Load Articles and Score Sentiment (VADER)

VADER is a lexicon-based sentiment analyzer that works well on short financial text.
We score the **headline + summary** (not full_text) — fast, and captures the signal.

In [ ]:
articles = pd.read_csv(FILES_DIR / 'q4_articles' / 'raw_articles_q4_2025.csv', parse_dates=['date_utc'])

# Filter to Q4 2025
articles = articles[articles['date_utc'] >= '2025-10-01'].copy()
articles['date'] = pd.to_datetime(articles['date_utc']).dt.normalize()

# Normalize ticker (articles use GOOGL, market uses GOOG)
articles['ticker'] = articles['ticker'].replace({'GOOGL': 'GOOG'})

print('Q4 articles:', len(articles))
print(articles['ticker'].value_counts())
articles[['ticker', 'date', 'headline', 'source_tier']].head()

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def score_article(row):
    # Combine headline and summary for richer signal
    headline = str(row['headline']) if pd.notna(row['headline']) else ''
    summary = str(row['summary']) if pd.notna(row['summary']) else ''
    text = f"{headline}. {summary}"
    scores = analyzer.polarity_scores(text)
    return scores['compound']  # -1 (very negative) to +1 (very positive)

tqdm.pandas(desc='Scoring articles')
articles['sentiment_vader'] = articles.progress_apply(score_article, axis=1)

print('\nSentiment distribution:')
print(articles['sentiment_vader'].describe())

# Quick sanity check
articles[['headline', 'sentiment_vader']].sort_values('sentiment_vader').head(3)

In [ ]:
articles[['headline', 'sentiment_vader']].sort_values('sentiment_vader', ascending=False).head(3)

## 3. Load Articles and Score Sentiment (FinBERT)

In [ ]:
articles = pd.read_csv(FILES_DIR / 'q4_articles' / 'raw_articles_q4_2025.csv', parse_dates=['date_utc'])

# Filter to Q4 2025
articles = articles[articles['date_utc'] >= '2025-10-01'].copy()
articles['date'] = pd.to_datetime(articles['date_utc']).dt.normalize()

# Normalize ticker (articles use GOOGL, market uses GOOG)
articles['ticker'] = articles['ticker'].replace({'GOOGL': 'GOOG'})

print('Q4 articles:', len(articles))
print(articles['ticker'].value_counts())
articles[['ticker', 'date', 'headline', 'source_tier']].head()

In [ ]:
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# map class indices to labels (0 = neutral, 1 = pos, -1 = neg)
labels = model.config.id2label 
sent_map = {0: 0, 1: 1, 2: -1}

In [ ]:
def score_articles_finbert(df, batch_size=16):
    # Prepare text: Combine headline and summary as you did with VADER
    df['combined_text'] = df['headline'].fillna('') + ". " + df['summary'].fillna('')
    texts = df['combined_text'].tolist()
    all_scores = []

    for i in tqdm(range(0, len(texts), batch_size), desc="FinBERT Scoring"):
        batch_texts = texts[i : i + batch_size]
        
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)
            # Map predictions to 1, 0, -1
            scores = [sent_map[p.item()] for p in predictions]
            all_scores.extend(scores)

    return all_scores

# Run the scoring
articles['sentiment_score'] = score_articles_finbert(articles)

## 4. Aggregate Sentiment to Daily Level

In [ ]:
def aggregate_daily_sentiment(df):
    agg = df.groupby(['ticker', 'date'])['sentiment_vader'].agg(
        article_count='count',
        sentiment_mean='mean',
        sentiment_max='max',
        sentiment_min='min',
        sentiment_std='std',
    ).reset_index()
    
    # Positive/negative counts
    pos = df[df['sentiment_vader'] > 0.05].groupby(['ticker', 'date']).size().reset_index(name='positive_count')
    neg = df[df['sentiment_vader'] < -0.05].groupby(['ticker', 'date']).size().reset_index(name='negative_count')
    
    agg = agg.merge(pos, on=['ticker', 'date'], how='left')
    agg = agg.merge(neg, on=['ticker', 'date'], how='left')
    agg[['positive_count', 'negative_count']] = agg[['positive_count', 'negative_count']].fillna(0).astype(int)
    agg['sentiment_std'] = agg['sentiment_std'].fillna(0)
    
    return agg

daily_sentiment = aggregate_daily_sentiment(articles)

print('Daily sentiment rows:', len(daily_sentiment))
daily_sentiment.head()

### LLM version

In [ ]:
def aggregate_daily_sentiment_finbert(df):
    agg = df.groupby(['ticker', 'date'])['sentiment_score'].agg(
        article_count='count',
        sentiment_mean='mean',
        sentiment_max='max',
        sentiment_min='min',
        sentiment_std='std',
    ).reset_index()
    
    # Positive/negative counts
    pos = df[df['sentiment_score'] == 1].groupby(['ticker', 'date']).size().reset_index(name='positive_count')
    neg = df[df['sentiment_score'] == -1].groupby(['ticker', 'date']).size().reset_index(name='negative_count')
    
    agg = agg.merge(pos, on=['ticker', 'date'], how='left')
    agg = agg.merge(neg, on=['ticker', 'date'], how='left')
    
    agg[['positive_count', 'negative_count']] = agg[['positive_count', 'negative_count']].fillna(0).astype(int)
    agg['sentiment_std'] = agg['sentiment_std'].fillna(0)
    
    # Net sentiment
    agg['net_sentiment_score'] = (agg['positive_count'] - agg['negative_count']) / agg['article_count']
    
    return agg

daily_sentiment = aggregate_daily_sentiment_finbert(articles)

## 5. Merge Market + Sentiment

In [ ]:
daily = market.merge(daily_sentiment, on=['ticker', 'date'], how='left')

# Fill sentiment columns with 0 on days with no articles
sentiment_cols = ['article_count', 'sentiment_mean', 'sentiment_max', 'sentiment_min',
                  'sentiment_std', 'positive_count', 'negative_count']
daily[sentiment_cols] = daily[sentiment_cols].fillna(0)

daily = daily.sort_values(['ticker', 'date']).reset_index(drop=True)

print('Final dataset shape:', daily.shape)
print('Columns:', list(daily.columns))
daily.head()

## 6. Quality Check

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Daily Features Overview — Q4 2025', fontsize=14)

for i, ticker in enumerate(TICKERS):
    t = daily[daily['ticker'] == ticker]
    
    ax1 = axes[i, 0]
    ax1.plot(t['date'], t['close'], label='close', color='steelblue')
    ax1.set_title(f'{ticker} — Close Price')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax1.grid(alpha=0.3)
    
    ax2 = axes[i, 1]
    ax2.bar(t['date'], t['article_count'], color='darkorange', alpha=0.7, label='articles')
    ax2_twin = ax2.twinx()
    ax2_twin.plot(t['date'], t['sentiment_mean'], color='green', linewidth=1.2, label='sentiment mean')
    ax2_twin.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax2.set_title(f'{ticker} — Article Count & Sentiment')
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plot to data/dataset_overview.png')

In [ ]:
# Check for missing values
print('Missing values:')
print(daily.isnull().sum())
print()
print('Rows per ticker:')
print(daily['ticker'].value_counts().sort_index())
print()
print('Days with 0 articles per ticker:')
print(daily.groupby('ticker')['article_count'].apply(lambda x: (x == 0).sum()))

## 7. Save Dataset

In [ ]:
# Rebuild the final dataset using the reusable session-aware pipeline
ticker_file_map = {
    'NVDA': FILES_DIR / 'NVDA_q4_2025.csv',
    'GOOG': FILES_DIR / 'GOOG_q4_2025.csv',
    'TSLA': FILES_DIR / 'TSLA_q4_2025.csv',
}
paths = DatasetPaths(
    market_dir=FILES_DIR,
    articles_csv=FILES_DIR / 'q4_articles' / 'raw_articles_q4_2025.csv',
    output_daily=DATA_DIR / 'daily_features.csv',
    output_article_sentiments=DATA_DIR / 'article_sentiments.csv',
)

daily, article_sentiments = build_dataset(paths, ticker_file_map)
daily.to_csv(paths.output_daily, index=False)
print(f'Saved {len(daily)} rows to {paths.output_daily}')
print()
print('Column descriptions:')
col_descriptions = {
    'date': 'Trading date',
    'ticker': 'Stock ticker (NVDA, GOOG, TSLA)',
    'return_1d': '1-day close-to-close return',
    'return_3d': '3-day close-to-close return',
    'rolling_volatility_5d': '5-day rolling std of daily returns',
    'article_count': 'Articles aligned to the trading session',
    'sentiment_mean': 'Mean VADER compound score across aligned articles',
    'sentiment_abs_mean': 'Average absolute sentiment magnitude',
    'sentiment_range': 'Daily max minus min sentiment',
    'sentiment_balance': 'Net positive-vs-negative article balance',
    'sentiment_mean_lag_1d': 'Prior-session mean sentiment',
    'sentiment_mean_lag_3d': '3-session lagged mean sentiment',
    'premarket_article_count': 'Articles published before 9:30 ET',
    'intraday_article_count': 'Articles published between 9:30 and 16:00 ET',
    'after_hours_article_count': 'Articles published after 16:00 ET and shifted forward',
}
for col, desc in col_descriptions.items():
    print(f'  {col:<28} {desc}')


In [ ]:
article_sentiments.to_csv(paths.output_article_sentiments, index=False)
print('Saved article-level sentiments to data/article_sentiments.csv')
article_sentiments.head()
